<h1>Table of Contents<span class="tocSkip"></span></h1>
<div class="toc"><ul class="toc-item"><li><span><a href="#Clean-up-ICA-object" data-toc-modified-id="Clean-up-ICA-object-1"><span class="toc-item-num">1&nbsp;&nbsp;</span>Clean up ICA object</a></span></li><li><span><a href="#Some-dictionaries-for-relating-regulators" data-toc-modified-id="Some-dictionaries-for-relating-regulators-2"><span class="toc-item-num">2&nbsp;&nbsp;</span>Some dictionaries for relating regulators</a></span></li><li><span><a href="#Largest-Digraph" data-toc-modified-id="Largest-Digraph-3"><span class="toc-item-num">3&nbsp;&nbsp;</span>Largest Digraph</a></span></li></ul></div>

In [1]:
from pymodulon.core import IcaData
from pymodulon.io import *
import itertools    

In [2]:
from graphviz import Digraph

In [3]:
ica = load_json_model('C:\\Users\\99hee\\precise_mg1655_R1\\data\\p_mg1655\\PRECISE_WT_MG1655_final2.json.gz')

## Clean up ICA object

In [4]:
# # glnG --> ntrC
ica.gene_table.gene_name['b3868'] = 'ntrC'

ica.gene_table.gene_name['b2921'] = 'srsR'

C:\Users\99hee\AppData\Local\Temp\ipykernel_15848\2239025944.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ica.gene_table.gene_name['b3868'] = 'ntrC'
C:\Users\99hee\AppData\Local\Temp\ipykernel_15848\2239025944.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ica.gene_table.gene_name['b2921'] = 'srsR'


## Some dictionaries for relating regulators

In [5]:
im2regbnum = {}
reg_name_dict = {'FlhDC':'FlhD', 
                 'IHF':'IhfA', 
                 'H-NS':'hns', 
                 'RcsAB':'RcsA'}
bad_regs = set()

for k, row in ica.imodulon_table.iterrows():
    reg_list = []
    if isinstance(row.regulator, str):
        for r in row.regulator.replace('/', '+').split('+'):
            try: 
                reg_list += [ica.name2num(reg_name_dict.get(r, r))]
            except:
                bad_regs = bad_regs.union(set([r]))
    im2regbnum[k] = reg_list

In [6]:
regbnum2im = {}

for k, gene_list in im2regbnum.items():
    for g in gene_list:
        if g in regbnum2im.keys():
            regbnum2im[g] = regbnum2im[g] + [k]
        else:
            regbnum2im[g] = [k]

## Largest Digraph

In [7]:
iM_names = ica.imodulon_table.loc[ica.imodulon_table.single_gene != 'True'].index.values

In [8]:

# generates network graph linking ALL regulators in network iM table to an iM, and linking iMs with shared genes
# only non-SG iMs included
dot = Digraph('iM TRN')
all_edges = pd.DataFrame(columns = ['source', 'target', 'interaction_type'])

# first, add all the nodes
"""
for k in ica.imodulon_names:
    dot.node(k, color = '#1f77b4')
"""
  
existing_edges = []
    
iM_names = ica.imodulon_table.loc[ica.imodulon_table.enrichment_category == 'regulatory'].index.values
iM_pairs = list(itertools.combinations(iM_names, 2))

for k in iM_names:
    dot.node(k, color='#1f77b4')

for pair in iM_pairs:
    k1 = pair[0]
    k2 = pair[1]
    genes1 = ica.view_imodulon(k1).index.values
    genes2 = ica.view_imodulon(k2).index.values

    for g1 in genes1:
        if g1 in genes2:

            # add links from iM to iM
            dot.edge(k1, k2, dir = 'none')
            all_edges = all_edges.append({'source':k1, 'target':k2,
                                          'interaction_type': 'shared_gene'},
                                        ignore_index = True)
            break
    
for g in regbnum2im.keys():
    for target_im in regbnum2im[g]:

        # add links from regulator to other iM
        edge_tuple = (ica.num2name(g), target_im)

        if not(edge_tuple in existing_edges):
            dot.edge(ica.num2name(g), target_im)
            all_edges = all_edges.append({'source':ica.num2name(g), 'target':target_im,
                                  'interaction_type': 'regulator_iM'},
                                        ignore_index = True)

dot = dot.unflatten()
dot.view()

C:\Users\99hee\AppData\Local\Temp\ipykernel_15848\349204064.py:31: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  all_edges = all_edges.append({'source':k1, 'target':k2,
C:\Users\99hee\AppData\Local\Temp\ipykernel_15848\349204064.py:31: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  all_edges = all_edges.append({'source':k1, 'target':k2,
C:\Users\99hee\AppData\Local\Temp\ipykernel_15848\349204064.py:31: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  all_edges = all_edges.append({'source':k1, 'target':k2,
C:\Users\99hee\AppData\Local\Temp\ipykernel_15848\349204064.py:31: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  all_edges = all_edges.ap

C:\Users\99hee\AppData\Local\Temp\ipykernel_15848\349204064.py:31: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  all_edges = all_edges.append({'source':k1, 'target':k2,
C:\Users\99hee\AppData\Local\Temp\ipykernel_15848\349204064.py:31: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  all_edges = all_edges.append({'source':k1, 'target':k2,
C:\Users\99hee\AppData\Local\Temp\ipykernel_15848\349204064.py:31: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  all_edges = all_edges.append({'source':k1, 'target':k2,
C:\Users\99hee\AppData\Local\Temp\ipykernel_15848\349204064.py:31: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  all_edges = all_edges.ap

C:\Users\99hee\AppData\Local\Temp\ipykernel_15848\349204064.py:31: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  all_edges = all_edges.append({'source':k1, 'target':k2,
C:\Users\99hee\AppData\Local\Temp\ipykernel_15848\349204064.py:31: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  all_edges = all_edges.append({'source':k1, 'target':k2,
C:\Users\99hee\AppData\Local\Temp\ipykernel_15848\349204064.py:31: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  all_edges = all_edges.append({'source':k1, 'target':k2,
C:\Users\99hee\AppData\Local\Temp\ipykernel_15848\349204064.py:31: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  all_edges = all_edges.ap

C:\Users\99hee\AppData\Local\Temp\ipykernel_15848\349204064.py:44: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  all_edges = all_edges.append({'source':ica.num2name(g), 'target':target_im,
C:\Users\99hee\AppData\Local\Temp\ipykernel_15848\349204064.py:44: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  all_edges = all_edges.append({'source':ica.num2name(g), 'target':target_im,
C:\Users\99hee\AppData\Local\Temp\ipykernel_15848\349204064.py:44: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  all_edges = all_edges.append({'source':ica.num2name(g), 'target':target_im,
C:\Users\99hee\AppData\Local\Temp\ipykernel_15848\349204064.py:44: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future ve

C:\Users\99hee\AppData\Local\Temp\ipykernel_15848\349204064.py:44: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  all_edges = all_edges.append({'source':ica.num2name(g), 'target':target_im,
C:\Users\99hee\AppData\Local\Temp\ipykernel_15848\349204064.py:44: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  all_edges = all_edges.append({'source':ica.num2name(g), 'target':target_im,
C:\Users\99hee\AppData\Local\Temp\ipykernel_15848\349204064.py:44: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  all_edges = all_edges.append({'source':ica.num2name(g), 'target':target_im,
C:\Users\99hee\AppData\Local\Temp\ipykernel_15848\349204064.py:44: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future ve

C:\Users\99hee\AppData\Local\Temp\ipykernel_15848\349204064.py:44: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  all_edges = all_edges.append({'source':ica.num2name(g), 'target':target_im,
C:\Users\99hee\AppData\Local\Temp\ipykernel_15848\349204064.py:44: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  all_edges = all_edges.append({'source':ica.num2name(g), 'target':target_im,
C:\Users\99hee\AppData\Local\Temp\ipykernel_15848\349204064.py:44: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  all_edges = all_edges.append({'source':ica.num2name(g), 'target':target_im,
C:\Users\99hee\AppData\Local\Temp\ipykernel_15848\349204064.py:44: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future ve

C:\Users\99hee\AppData\Local\Temp\ipykernel_15848\349204064.py:44: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  all_edges = all_edges.append({'source':ica.num2name(g), 'target':target_im,


'iM TRN.gv.pdf'

In [37]:
# all_edges.to_csv('network_edges_gene_overlap_and_regulator.csv')

In [12]:
# generates network graph linking only regulators in iMs in network iM table to an iM, and linking iMs with shared genes
# only non-SG iMs included
dot = Digraph('iM TRN')
all_edges = pd.DataFrame(columns = ['source', 'target', 'interaction_type'])

# first, add all the nodes
"""
for k in ica.imodulon_names:
    dot.node(k, color = '#1f77b4')
"""

existing_edges = []

    
iM_names = ica.imodulon_table.loc[ica.imodulon_table.enrichment_category == 'regulatory'].index.values
iM_pairs = list(itertools.combinations(iM_names, 2))

for k in iM_names:
    dot.node(k, color='#1f77b4')

for pair in iM_pairs:
    k1 = pair[0]
    k2 = pair[1]
    genes1 = ica.view_imodulon(k1).index.values
    genes2 = ica.view_imodulon(k2).index.values

    for g1 in genes1:
        if g1 in genes2:

            # add links from iM to iM
            dot.edge(k1, k2, dir = 'none')
            all_edges = all_edges.append({'source':k1, 'target':k2,
                                          'interaction_type': 'shared_gene'},
                                        ignore_index = True)
            break
    
for g in regbnum2im.keys():
    iMs_with_gene = ica.imodulons_with(g)
    iMs_with_gene = [x for x in iMs_with_gene if not 'SG_' in x]
    if len(iMs_with_gene) > 0:
        for target_im in regbnum2im[g]:

            # add links from regulator to other iM
            edge_tuple = (ica.num2name(g), target_im)

            if not(edge_tuple in existing_edges):
                dot.edge(ica.num2name(g), target_im)
                all_edges = all_edges.append({'source':ica.num2name(g), 'target':target_im,
                                      'interaction_type': 'regulator_iM'},
                                            ignore_index = True)

dot = dot.unflatten()
dot.view()

C:\Users\99hee\AppData\Local\Temp\ipykernel_18300\3102149260.py:32: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  all_edges = all_edges.append({'source':k1, 'target':k2,
C:\Users\99hee\AppData\Local\Temp\ipykernel_18300\3102149260.py:32: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  all_edges = all_edges.append({'source':k1, 'target':k2,
C:\Users\99hee\AppData\Local\Temp\ipykernel_18300\3102149260.py:32: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  all_edges = all_edges.append({'source':k1, 'target':k2,
C:\Users\99hee\AppData\Local\Temp\ipykernel_18300\3102149260.py:32: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  all_edges = all_edge

C:\Users\99hee\AppData\Local\Temp\ipykernel_18300\3102149260.py:32: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  all_edges = all_edges.append({'source':k1, 'target':k2,
C:\Users\99hee\AppData\Local\Temp\ipykernel_18300\3102149260.py:32: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  all_edges = all_edges.append({'source':k1, 'target':k2,
C:\Users\99hee\AppData\Local\Temp\ipykernel_18300\3102149260.py:32: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  all_edges = all_edges.append({'source':k1, 'target':k2,
C:\Users\99hee\AppData\Local\Temp\ipykernel_18300\3102149260.py:32: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  all_edges = all_edge

C:\Users\99hee\AppData\Local\Temp\ipykernel_18300\3102149260.py:32: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  all_edges = all_edges.append({'source':k1, 'target':k2,
C:\Users\99hee\AppData\Local\Temp\ipykernel_18300\3102149260.py:32: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  all_edges = all_edges.append({'source':k1, 'target':k2,
C:\Users\99hee\AppData\Local\Temp\ipykernel_18300\3102149260.py:32: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  all_edges = all_edges.append({'source':k1, 'target':k2,
C:\Users\99hee\AppData\Local\Temp\ipykernel_18300\3102149260.py:32: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  all_edges = all_edge

C:\Users\99hee\AppData\Local\Temp\ipykernel_18300\3102149260.py:48: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  all_edges = all_edges.append({'source':ica.num2name(g), 'target':target_im,
C:\Users\99hee\AppData\Local\Temp\ipykernel_18300\3102149260.py:48: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  all_edges = all_edges.append({'source':ica.num2name(g), 'target':target_im,
C:\Users\99hee\AppData\Local\Temp\ipykernel_18300\3102149260.py:48: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  all_edges = all_edges.append({'source':ica.num2name(g), 'target':target_im,
C:\Users\99hee\AppData\Local\Temp\ipykernel_18300\3102149260.py:48: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a futur

'iM TRN.gv.pdf'

In [9]:
# all_edges.to_csv('network_edges_gene_overlap_and_iM_regulators_only.csv')